In [0]:
USE CATALOG olist;
USE SCHEMA silver;

-- ============================================================
-- Key strategy: natural keys from source data are used as
-- primary keys in silver (they're already unique per Olist's
-- data dictionary). No surrogate keys needed yet — those are
-- introduced in gold for the dimensional model.
-- Duplicates: handled at transform time (silver_to_gold script)
-- using deterministic dedup logic (row_number over natural key,
-- keeping the latest / first occurrence as documented per table).
-- ============================================================

CREATE TABLE IF NOT EXISTS customers (
  customer_unique_id       STRING NOT NULL,
  customer_zip_code_prefix INT,
  customer_city            STRING,
  customer_state           STRING,
  _loaded_at                TIMESTAMP,
  _source                   STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS customer_orders (
  customer_id        STRING NOT NULL,
  customer_unique_id STRING NOT NULL,
  _loaded_at          TIMESTAMP,
  _source             STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS orders (
  order_id                      STRING NOT NULL,
  customer_id                   STRING NOT NULL,
  order_status                  STRING,
  order_purchase_timestamp      TIMESTAMP,
  order_approved_at             TIMESTAMP,
  order_delivered_carrier_date  TIMESTAMP,
  order_delivered_customer_date TIMESTAMP,
  order_estimated_delivery_date TIMESTAMP,
  _loaded_at                     TIMESTAMP,
  _source                        STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS order_items (
  order_id            STRING NOT NULL,
  order_item_id        INT NOT NULL,
  product_id           STRING,
  seller_id            STRING,
  shipping_limit_date  TIMESTAMP,
  price                DECIMAL(10,2),
  freight_value        DECIMAL(10,2),
  _loaded_at            TIMESTAMP,
  _source               STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS payments (
  order_id             STRING NOT NULL,
  payment_sequential   INT NOT NULL,
  payment_type         STRING,
  payment_installments INT,
  payment_value        DECIMAL(10,2),
  _loaded_at            TIMESTAMP,
  _source               STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS reviews (
  review_id             STRING NOT NULL,
  order_id              STRING NOT NULL,
  review_score          INT,
  review_comment_title  STRING,
  review_comment_message STRING,
  review_creation_date  TIMESTAMP,
  review_answer_timestamp TIMESTAMP,
  _loaded_at             TIMESTAMP,
  _source                STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS products (
  product_id               STRING NOT NULL,
  product_category_name    STRING,
  product_name_length      INT,
  product_description_length INT,
  product_photos_qty       INT,
  product_weight_g         DECIMAL(10,2),
  product_length_cm        DECIMAL(10,2),
  product_height_cm        DECIMAL(10,2),
  product_width_cm         DECIMAL(10,2),
  _loaded_at                TIMESTAMP,
  _source                   STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS product_categories (
  product_category_name          STRING NOT NULL,
  category_name_english           STRING,
  _loaded_at                       TIMESTAMP,
  _source                          STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sellers (
  seller_id               STRING NOT NULL,
  seller_zip_code_prefix  INT,
  seller_city             STRING,
  seller_state            STRING,
  _loaded_at               TIMESTAMP,
  _source                  STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS geolocation (
  geolocation_zip_code_prefix INT NOT NULL,
  geolocation_lat              DOUBLE,
  geolocation_lng              DOUBLE,
  geolocation_city             STRING,
  geolocation_state            STRING,
  _loaded_at                    TIMESTAMP,
  _source                       STRING
)
USING DELTA;